# *SARGASSUM* GROWTH DURING TRANSPORT 

This notebook guides you through the process of modelling growth of *Sargassum* rafts during transport in the Atlantic Ocean. The aim of this project was to create a customizable *Sargassum* growth model to be included in Lagrangian simulations using **Parcels**. Customizability is essential as emperical studies report varying maximal growth rates and types of physicochemical limitations. Additionally, three genotypes of *Sargassum* exist in Atlantic Ocean which seem to behave differently. Variables as maximum growth rate and optimal temperature can therefore be easily adapted in this growth model. The notebook is build up by the following steps:
- Defining the limitation functions on growth by physico-chemical factors
- Creating a map of Sargassum locations based on satellite detections from the Sargassum Watch System (SaWS)
- Combining all the required data fields for *Sargassum* simulation as a fieldset
- Defining *Sargassum* rafts as particles for a simulation with related characteristic variables
- Setting up the operational kernels to describe *Sargassum* behaviour
- Executing the simulation
- Visualizing the output

We start by importing the relevant packages.


In [ ]:
#Importing relevent packages
import parcels
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import erfc
import cartopy.crs as ccrs
import cartopy.feature
import cmocean.cm as cmo
from datetime import datetime, timedelta

## Defining the limitation functions on growth by physico-chemical factors

The growth of _Sargassum_ is known to be limited by for example oceanic conditions and nutrient availability. Some of the most important factors limiting Sargassum growth are temperature, salinity and nitrogen availability. The reducing effect on growth by these factors implemented in this model as **growth factors**, ranging between 0 (meaning no growth and maximum limitation) and 1 (meaning maximum growth and no limitation). A lot of uncertainty remains about the exact dependencies of Sargassum on these parameters. Tweeking these model parameters can alter the change in biomass.

In [ ]:
#RATES
maximum_growth_rate = 0.095 #doublings/day (Magana-Gallegos et al., 2023)
mortality_rate = 0.025      #relative loss/day

#Minimum, maximum and optimal temperature
T_min = 20      #degC (Jouanno et al., 2025)
T_max = 31      #degC (Jouanno et al., 2025)
T_opt = 27.5    #degC (Jouanno et al., 2025)

#Nitrogen half saturation constant
k_N  = 0.001 #mmol/m3

#Optimal salinity
S_opt = 36 #psu (Jouanno et al., 2025)

The figure below visualizes the dependencies of Sargassum growth on the various physico-chemical factors, which are determined by the model paramters selected in the previous cell.

In [ ]:
#Variable initialization
T = np.arange(14,38,0.5)
N = np.arange(0.00001,0.1,0.00001)
S = np.arange(20,40,0.05)

#Growth factor curves
T_x = np.full_like(T,T_min)
T_x[T > T_opt] = T_max
growth_factor_T = np.exp(-2 * ((T - T_opt) /(T_x - T_opt))**2 )
growth_factor_N = N / (N + k_N)
growth_factor_S = np.exp(-0.02 * (S_opt - S)**2)

#Figure
fig_growthcurves, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(10, 7))

#Growth rate & mortality rate
rates = [maximum_growth_rate, mortality_rate]
labels = ['Max. growth rate', 'Mortality rate']
ax1.bar(labels, rates, color=['olivedrab', 'goldenrod'], alpha=0.7)
ax1.set_ylabel('Rate')
ax1.set_title('Growth & mortality rate')
ax1.grid(alpha=0.3, linestyle='--', axis='y')

#Temperature
ax2.plot(T, growth_factor_T, label=f'Jouanno et al. (2025)', c = 'cadetblue')
ax2.set_xlabel('Temperature [°C]')
ax2.set_ylabel('Growth factor')
ax2.set_title('Dependency on temperature')
ax2.grid(alpha=0.3, linestyle='--')
ax2.legend(loc='upper left')
ax2.set_xlim(15,37)

#Nitrogen concentration
ax3.plot(N, growth_factor_N, label = f'Monod equation with k$_N$= {k_N}', color='indianred')
ax3.set_xlabel('Nitrate concentration [$mmol / m^3$]')
ax3.set_ylabel('Growth factor')
ax3.set_title(f'Dependency on NO$_3$ concentration')
ax3.legend(loc='lower right')
ax3.grid(alpha=0.3, linestyle='--')
ax3.set_xlim(-0.001,0.1)

#Salinity concentration
ax4.plot(S, growth_factor_S, label = f' Jouanno et al. (2025)', color='purple')
ax4.set_xlabel('Salinity [$psu$]')
ax4.set_ylabel('Growth factor')
ax4.set_title(f'Dependency on salinity' )
ax4.legend()
ax4.grid(alpha=0.3, linestyle='--')
ax4.set_xlim(20,40)

plt.tight_layout()
plt.show()

## Creating a map of *Sargassum* locations based on satellite detections


To initialize Sargassum particles at locations where actual Sargassum has been detected, we use satellite images of the Sargassum Watch System (SaWS) of the University of South-Florida.  We use the composite FA-UNET-DENSITY-7DAY images, which show the floating algae (FA) density, in terms of percentage of area cover. A value of 0.1 on the color bar indicates 0.1\% surface area coverage by floating algae in that location. The FA density is calculated as a mean of the 7 past days, and is based on the U-Net method described by Hu et al. (2023). The function in the following cell was created to create a Sargassum release grid based on these SaWS satellite images.

In [ ]:
def sarg_grid_from_sat(image_name, north, south, east, west, coarse=False):

    #Loading the image as an RGB array
    from PIL import Image
    img = Image.open(image_name).convert("RGB")
    img_array = np.array(img)

    #Computing brightness by approximating average or weighted sum of RGB
    brightness = img_array.mean(axis=2)

    #Creating a land mask based on the brown color of landmass
    r, g, b = img_array[:,:,0], img_array[:,:,1], img_array[:,:,2]

    #Brown tends to be dark, reddish, and not too saturated
    land_mask = (
        (r > 60) & (r < 160) &          # moderate red
        (g > 30) & (g < 110) &          # moderate green
        (b < 70) &                      # low blue
        (brightness < 120) &            # exclude bright oranges
        ((r - g) > 15) &                # red clearly higher than green
        ((r - b) > 40)                  # red much higher than blue
        )
    #Expanding land mask with binary_dilation method by 20 pixels (~20 km as 1 pixel ~ 1 km resolution)
    from scipy.ndimage import binary_dilation
    expanded_land_mask = binary_dilation(land_mask, iterations=20)

    #Setting threshold to get binary mask - set at 60 to include bright pixels and especially also red pixels
    threshold = 60
    binary_mask = (brightness > threshold).astype(int)

    #Applying the expaned land mask on binary mask
    binary_mask[expanded_land_mask] = 0

    #Creating coordinate grids based on bounding boxes
    height, width = binary_mask.shape
    lats = np.linspace(north, south, height)
    lons = np.linspace(west, east, width)

    #Creating 2D coordinate grids
    lon_grid, lat_grid = np.meshgrid(lons, lats)

    #Downsample with stride 2 (or other number) if you want to select less particles
    stride = 2 if coarse else 1
    mask = binary_mask[::stride, ::stride]
    lat = lat_grid[::stride, ::stride]
    lon = lon_grid[::stride, ::stride]

    amount = int(mask.sum())

    #Creating 2D grids of Sargassum release locations
    sarg_lon_grid = xr.DataArray(lon).where(mask == 1)
    sarg_lat_grid = xr.DataArray(lat).where(mask == 1)

    print('Shape of grid:', np.shape(sarg_lon_grid))

    #To prepare the grids as ParticleSet input, NaNs are removed and and arrays are ravelled (flattened)
    no_nan_mask = (~np.isnan(sarg_lon_grid)) & (~np.isnan(sarg_lat_grid))
    sarg_lon_grid = sarg_lon_grid.values[no_nan_mask].ravel()
    sarg_lat_grid = sarg_lat_grid.values[no_nan_mask].ravel()
    print('Reshaped grid as particle set:', np.shape(sarg_lon_grid))

    return sarg_lon_grid, sarg_lat_grid, amount

The SaWS data is split up in images with different regions. In this notebook, we focus on the Atlantic basin in July 2024 and therefore we use images from the Central and Central East Atlantic with the following coordinates (N, S, E, W):
- The Central Atlantic box is bounded by: 22.0, 0.0, -38.0, -63.0
- The Centrel East Atlantic box is bounded by: 22.0, 0.0, -11.5, -38.0

The images below will be used as input and show the two areas with composite data from 01-07-2024. Black pixels indicate that there is no data, at that location (mostly due to couds).

<p float="left">
  <img src="/nethome/6903894/testing/Input_data_test/C20241772024183.1KM.C_ATLANTIC.7DAY.L3D.FA_UNET_DENSITY.png" height="330" />
  <img src="/nethome/6903894/testing/Input_data_test/C20241772024183.1KM.CE_ATLANTIC.7DAY.L3D.FA_UNET_DENSITY.png" height="330" />
</p>


In the following piece of code, we load the images and apply the `sarg_grid_from_sat` function defined in the previous cell. Make sure the correct coordinates of the bounding boxes are selected. You can also choose if you want the grid to be `coarse`, so the simulation runs faster. Also, the grids are combined to be suitable as input for the `ParticleSet`, later in the notebook.

In [ ]:
#Loading satellite images
image_name_Central_Atlantic = "/Users/erik/Desktop/FromElena/6903894/testing/Input_data_test/C20241772024183.1KM.C_ATLANTIC.7DAY.L3D.FA_UNET_DENSITY.png"
image_name_Central_East_Atlantic = "/Users/erik/Desktop/FromElena/6903894/testing/Input_data_test/C20241772024183.1KM.CE_ATLANTIC.7DAY.L3D.FA_UNET_DENSITY.png"

#Applying function on images
sarg_lon_grid_C, sarg_lat_grid_C, amount_C = sarg_grid_from_sat(image_name_Central_Atlantic, 22.0, 0.0, -38.0, -63.0, coarse=True)
sarg_lon_grid_CE, sarg_lat_grid_CE, amount_CE = sarg_grid_from_sat(image_name_Central_East_Atlantic, 22.0, 0.0, -11.5, -38.0, coarse=True)

#Adding the grids of both images to 1 longitude array and 1 latitude array and 1 amount
sarg_LONGITUDES = np.append(sarg_lon_grid_C, sarg_lon_grid_CE, axis=0)
sarg_LATITUDES  = np.append(sarg_lat_grid_C, sarg_lat_grid_CE, axis=0)
sarg_AMOUNT = amount_C + amount_CE

#Plotting to check the release locations
fig = plt.figure(figsize = (9,4), constrained_layout=True)
ax = plt.axes(projection=ccrs.PlateCarree(),zorder=4)
ax.add_feature(cartopy.feature.COASTLINE.with_scale('10m'),zorder=2)
ax.add_feature(cartopy.feature.LAND.with_scale('10m'), zorder=1)
ax.gridlines(draw_labels=['left','bottom'], zorder=0, alpha=0.3, linestyle='--')
ax.scatter(sarg_LONGITUDES, sarg_LATITUDES, s = 0.1, color='olivedrab', zorder=5)
ax.set_extent([-75,-12, -2,21])
ax.set_title(f'Satellite-based Sargassum release locations: {sarg_AMOUNT} particles')
plt.show()

## Combining all the required data fields for *Sargassum* simulation as a fieldset

To simulate Sargassum transport, we need to load all the data of the Atlantic Ocean domain as a `fieldset`. The following datasets and variables are required for the physical transport and growth model:
- **Physical transport** 
    - Current advection: `U` , `V`
    - Stokes drift: `U_wave_Stokes`, `V_wave_Stokes`, `wave_Tp`
    - Windage: `U_wind`, `V_wind`
- **Tracer fields**
    - Temperature: `T`
    - Salinity: `S`
    - Nitrate concentration: `no3`

First make sure the time-settings (corresponding to your satellite images) are correct. As default, duration of the simulations is set as 1 month to see the growth development. 

In [ ]:
#SIMULATION TIME SETTINGS
#Time when you want to start the simulation
year = 2024
month = 7
day = 1
#Amount of days that will be simulated
simulation_days = 5  # 30 TODO update to 30 days for final simulation

#Time-related settings
starttime = datetime(year, month, day)
release_times = np.array([starttime])
dtime_data = timedelta(days=1)              #time-resolution of input data files
endtime = release_times[-1] + timedelta(days=simulation_days) + dtime_data
dtime_execute = timedelta(minutes=10)       #integration step of simulation


Now, we create the fieldset. As we download quite some data, running this step will probably take a few minutes. 

In [ ]:
import copernicusmarine
import os
# TODO make this on-the-fly access instead of downloading the file first and then loading it

dirname = "/Users/erik/Desktop/FromElena/copernicus_marine_data"

DATASET_IDs = [
    "cmems_mod_glo_phy-cur_anfc_0.083deg_P1D-m",
    "cmems_mod_glo_phy-so_anfc_0.083deg_P1D-m",
    "cmems_mod_glo_phy-thetao_anfc_0.083deg_P1D-m",
    "cmems_mod_glo_bgc-nut_anfc_0.25deg_P1D-m",
    "cmems_mod_glo_wav_anfc_0.083deg_PT3H-i",
    "cmems_obs-wind_glo_phy_my_l4_0.125deg_PT1H",
]
variables = [["uo", "vo"], ["so"], ["thetao"], ["no3"], ["VSDX", "VSDY", "VTPK"], ["northward_wind", "eastward_wind"]]
filenames = ["cur.nc", "so.nc", "thetao.nc", "no3.nc", "stokes.nc", "wind.nc"]
for i in range(len(DATASET_IDs)):

    filename = f"{dirname}_{filenames[i]}"
    if os.path.exists(filename):
                print(f"File {filename} already exists, skipping...")
                continue

    copernicusmarine.subset(
        dataset_id=DATASET_IDs[i],
        variables=variables[i],
        minimum_longitude=-75,
        maximum_longitude=-10,
        minimum_latitude=-4,
        maximum_latitude=24,
        start_datetime="2024-07-01T00:00:00",
        end_datetime="2024-07-31T00:00:00",
        minimum_depth=0.5,
        maximum_depth=0.5,
        output_filename=filename,
    )

In [ ]:
fields = []

for i in range(len(filenames)):
    filename = f"{dirname}_{filenames[i]}"

    print(f"Loading {filename}")
    ds_fields = xr.open_mfdataset(filename, combine="by_coords")

    # TODO check how we can get good performance without loading full dataset in memory
    ds_fields.load()  # load the dataset into memory

    # TODO clean up this part, maybe make it more generic and less hardcoded
    if "cur" in filename:
        flds = {"U": ds_fields["uo"], "V": ds_fields["vo"]}
    elif "so" in filename:
        flds = {"so": ds_fields["so"]}
    elif "thetao" in filename:
        flds = {"thetao": ds_fields["thetao"]}
    elif "no3" in filename:
        flds = {"no3": ds_fields["no3"]}
    elif "stokes" in filename:
        ds_fields = ds_fields.expand_dims(dim={"depth": [0]})
        flds = {"VSDX": ds_fields["VSDX"], "VSDY": ds_fields["VSDY"], "VTPK": ds_fields["VTPK"]}
    elif "wind" in filename:
        ds_fields = ds_fields.expand_dims(dim={"depth": [0]})
        flds = {"northward_wind": ds_fields["northward_wind"], "eastward_wind": ds_fields["eastward_wind"]}

    ds_fset = parcels.convert.copernicusmarine_to_sgrid(fields=flds)
    fset = parcels.FieldSet.from_sgrid_conventions(ds_fset, mesh="spherical")

    for fld in fset.fields.values():
        fields.append(fld)
fieldset = parcels.FieldSet(fields)


In [ ]:
# making vector fields of wind and stokes
fieldset.add_field(parcels.VectorField("UVStokes", fieldset.VSDX, fieldset.VSDY, vector_interp_method=parcels.interpolators.XLinear_Velocity))
fieldset.add_field(parcels.VectorField("UVWind", fieldset.northward_wind, fieldset.eastward_wind, vector_interp_method=parcels.interpolators.XLinear_Velocity))

# setting NaN values to 0 for vector fields
for field in ["U", "V", "VSDX", "VSDY", "northward_wind", "eastward_wind"]:
    getattr(fieldset, field).data = getattr(fieldset, field).data.fillna(0)

# TODO check if no3 NaN setting also needed?
fieldset.no3.data = fieldset.no3.data.fillna(0)
fieldset.thetao.data = fieldset.thetao.data.fillna(0)
fieldset.so.data = fieldset.so.data.fillna(0)
fieldset.VTPK.data = fieldset.VTPK.data.fillna(0)

#setting other interpolators to XLinearInvdistLandTracer,
for field in [fieldset.so, fieldset.thetao, fieldset.no3, fieldset.VTPK]:
    field.interp_method = parcels.interpolators.XLinearInvdistLandTracer

In [ ]:
#TODO these can be removed as fieldset constants

fieldset.add_constant('G', 9.81)  # Gravitational constant [m s-1]

#Overall maximal growth rate (Corbin & Oxenford)
fieldset.add_constant('MGR_SF3', 0.124)
fieldset.add_constant('MGR_SN1', 0.083)
fieldset.add_constant('MGR_SN8', 0.053)
#Set initial weight
fieldset.add_constant('initial_weight', 50) #grams

## Defining *Sargassum* rafts as simulation particles with characteristic variables

Here, we start by defining the `class` of each released particle. The class `SargassumParticle` has a lot of particle-specific characteristics, all defined as a `parcels.Variable`. These characteristics can be different for each particle and can vary trough time of the simulation. 

Hereafter, the complete particleset (`pset`) is created, based on the defined `fieldset`, `pclass` and earlier satellite-based derived start longitudes and latitudes. The `depth` of the particles is 0, which corresponds to the upper extent of the raft. The lower extent is defined as the  particle specific variable `depth_extent`. 

Finally, the starting locations are plotted together with the zonal velocity field a a check. 

In [ ]:
SargassumParticle = parcels.Particle.add_variable(
    [
        parcels.Variable('temperature', dtype=np.float32, to_write=True, initial=0),
        parcels.Variable('salinity', dtype=np.float32, to_write=True, initial=0),
        parcels.Variable('depth_extent', dtype=np.float32, to_write=True, initial=1),
        parcels.Variable('nitrogen', dtype=np.float32, to_write=True, initial=0),
        parcels.Variable('biomass_SF3', dtype=np.float32, to_write=True, initial=1),
        parcels.Variable('biomass_SN1', dtype=np.float32, to_write=True, initial=1),
        parcels.Variable('biomass_SN8', dtype=np.float32, to_write=True, initial=1),
        parcels.Variable('biomass_loss', dtype=np.float32, to_write=True, initial=0),
        parcels.Variable('stranded', dtype=np.float32, to_write=True, initial=0),
        parcels.Variable('limitation', dtype=np.float32, to_write=True, initial=1),
        parcels.Variable('lim_salinity', dtype=np.float32, to_write=True, initial=0),
        parcels.Variable('lim_temp', dtype=np.float32, to_write=True, initial=0),
        parcels.Variable('lim_no3', dtype=np.float32, to_write=True, initial=0),
        parcels.Variable('speed_currents', dtype=np.float32, to_write=True, initial=0),
        parcels.Variable('speed_stokes', dtype=np.float32, to_write=True, initial=0),
        parcels.Variable('speed_wind', dtype=np.float32, to_write=True, initial=0),
        parcels.Variable('decay_factor', dtype=np.float32, to_write=True, initial=0),
        parcels.Variable('decay_averaged', dtype=np.float32, to_write=True, initial=0),
        parcels.Variable('decay_integrated_lower', dtype=np.float32, to_write=True, initial=0),
        parcels.Variable('decay_integrated_upper', dtype=np.float32, to_write=True, initial=0),
        parcels.Variable('wind_coefficient', dtype=np.float32, to_write=True, initial=0.01),
    ]
)

pset = parcels.ParticleSet(
    fieldset=fieldset,                  #Fieldsets on which the particles are advected
    pclass = SargassumParticle,         #Particle class
    lon = sarg_LONGITUDES,              #Vector of release longitudes
    lat = sarg_LATITUDES,               #Vector of release latitudes
    z = np.zeros_like(sarg_LONGITUDES)  #Vector of release depths
)
# nparticles = 4 * 4

# pset = parcels.ParticleSet.from_list(
#       fieldset=fieldset,  # the fields on which the particles are advected
#       pclass = SargassumParticle,
#       lon=[-43, -43, -40, -40,
#            -20, -20, -17, -17,
#            -55, -55, -52, -52,
#            -23, -23, -20, -20], # a vector of release longitudes

#       lat=[2,  5,  2, 5,
#            -2, 1, -2, 1,
#            13, 16, 13, 16,
#            9, 12, 9, 12],       # a vector of release latitudes

#       depth=[0] * nparticles    # a vector of release depths
#   )


#Plotting velocity field of first timestep
fig = plt.figure(figsize = (9,4), constrained_layout=True)
ax = plt.axes(projection=ccrs.PlateCarree())
vplot = ax.pcolormesh(fieldset.U.grid.lon, fieldset.U.grid.lat, fieldset.U.data[0, 0,:,:], vmin=-1, vmax=1, cmap=cmo.delta, transform=ccrs.PlateCarree())
lplot = ax.scatter(pset.lon, pset.lat, s =0.1, color='tab:red')
ax.add_feature(cartopy.feature.COASTLINE.with_scale('50m'))
ax.add_feature(cartopy.feature.LAND.with_scale('10m'))
ax.gridlines(draw_labels=['left','bottom'], zorder=0, alpha=0.3, linestyle='--')
ax.set_title(f'Start locations of particles and horizontal advection field at {starttime.date()}')
ax.set_extent([-75,-12, -2,21])
cbar = fig.colorbar(vplot, ax=ax, shrink = 0.5)
cbar.set_label('Zonal velocity [m/s]')
plt.show()

## Setting up the operational kernels to describe *Sargassum* behaviour

Kernels describe 'operations' that are performed on every particle at every timestep of the simulation. They allow. The kernels allow for behaviour of the particles. For transport by currents, we apply the build-in `AdvectionRK4` kernel, which is commonly used for 2D advection. For other processes, customized kernels were created which describe specific *Sargassum* raft behaviour. The following kernels :
1. `di_Stokes_drift`: Kernel for Stokes drift integrated between upper and lower extent of raft
2. `windage_drift`: Kernel for inclusion of leeway windage (relative)
3. `stranding`: Kernel that accounts for stranding 
4. `sargassum_biological_growth_model`: Kernel for calculation of relative biomass, dependent on temperature, nitrate availability and salinity. 

! Note that the `sargassum_biological_growth_model` requires certain growth model parameters regarding the limitation functions on growth by physico-chemical factors. To have them aligned with the first cell of this notebook, it should be checked if the values correspond. 

In [ ]:
def di_Stokes_drift(particles, fieldset):
    """Depth-integrated Stokes drift kernel:

    Description
    ----------
    Using the approach in [1] (assuming a Phillips wave spectrum), equation A.6 and A.7 of [2] are used to determine
    the Stokes drift velocity integrated over depth between upper extent and lower extent of particle.

    Stokes drift is treated as a linear addition to the velocity field.

    Parameter Requirements
    ----------
    fieldset :
        - fieldset.UVStokes: zonal and meridional Stokes drift velocity at surface [m s-1]
        - fieldset.VTPK: the peak wave period field [s].

    References
    ----------
    [1] Breivik (2016) - https://doi.org/10.1016/j.ocemod.2016.01.005
    [2] Li et al. (2017) - http://dx.doi.org/10.1016/j.ocemod.2017.03.016
    """

    delta_z = particles.depth_extent - particles.z
    z_up = particles.z
    z_low = z_up + particles.depth_extent

    #Sampling the U / V components of Stokes drift at upper level
    stokes_U, stokes_V = fieldset.UVStokes[particles]

    #Sampling the peak wave period
    T_p = fieldset.VTPK[particles]

    #Only computing displacements if the peak wave period is large enough and the particle is in the water
    ptcls_wave = particles[T_p > 1E-14] #& (particles.z < local_bathymetry)]

    #Peak wave frequency
    omega_p = 2. * np.pi / T_p

    #Peak wave number
    k_p = (omega_p ** 2) / fieldset.G

    #Decay function lower extent, based on Equation A.6 of Li et al. (2017)
    decay_function_lower = 1/(2*k_p) * (
                1 - np.exp(-2.0*k_p*z_low)
                - (2.0/3.0) * (1 + np.sqrt(np.pi) * (2.0*k_p*z_low)**(3.0/2.0) * erfc(np.sqrt(2.0*k_p*z_low))
                - (1 + 2.0*k_p*z_low) * np.exp(-2.0*k_p*z_low)   )
                )


    #Decay function upper extent, based on Equation A.6 of Li et al. (2017)
    decay_function_upper = 1/(2*k_p) * (
                1 - np.exp(-2.0*k_p*z_up)
                - (2.0/3.0) * (1 + np.sqrt(np.pi) * (2.0*k_p*z_up)**(3.0/2.0) * erfc(np.sqrt(2.0*k_p*z_up))
                - (1 + 2.0*k_p*z_up) * np.exp(-2.0*k_p*z_up)   )
                )

    #Integration function between surface and lower level based on Equation A.7 of Li et al. (2017)
    stokes_U_integrated = (stokes_U * decay_function_lower - stokes_U * decay_function_upper) / delta_z
    stokes_V_integrated = (stokes_V * decay_function_lower - stokes_V * decay_function_upper) / delta_z

    #Saving lower and upper decay function and total Stokes decay factor as particle variables
    ptcls_wave.decay_integrated_lower = decay_function_lower
    ptcls_wave.decay_integrated_upper = decay_function_upper
    ptcls_wave.decay_factor = (decay_function_lower - decay_function_upper) / delta_z

    #Compute particle displacement based on depth-integrated Stokes velocity
    ptcls_wave.dlon += stokes_U_integrated  * ptcls_wave.dt
    ptcls_wave.dlat += stokes_V_integrated  * ptcls_wave.dt

def windage_drift(particles, fieldset):
    """Leeway windage kernel.

    Description
    ----------
    A simple windage kernel that applies a linear relative 'wind velocity' to the particle.
    Slightly adapted for the usage for Sargassum.

    We treat the windage drift as a linear addition to the velocity field
        :math:`u(x,t) = u_c(x,t) + C_w * (u_w(x,t)-u_c(x,t))`
    where :math:`u_c` is the ocean current velocity, :math:`u_w` is the wind velocity
    at 10m height, and :math:`C_w` is the windage coefficient.

    For further description, see https://plastic.parcels-code.org/en/latest/physicskernels.html#wind-induced-drift-leeway

    Parameter Requirements
    ----------
    particle :
        - wind_coefficient - the particle windage coefficient in decimals.
    fieldset :
        - `fieldset.UV`, the ocean velocities. Units [m s-1].
        - `fieldset.UVWind`, the wind velocity field at 10m height above sea surface. Units [m s-1].

    Kernel Requirements
    ----------
    Order of Operations:
        None - can be applied at any time.

    """
    # Sample ocean velocities
    (ocean_U, ocean_V) = fieldset.UV[particles]

    # Use a basic approach to only apply windage to particle in the ocean
    wind_U, wind_V = fieldset.UVWind[particles]

    # Compute particle displacement
    particles.dlon += particles.wind_coefficient * (wind_U - ocean_U) * particles.dt
    particles.dlat += particles.wind_coefficient * (wind_V - ocean_V) * particles.dt

def stranding(particles, fieldset):
    """Data-based stranding kernel.

    Description
    ----------
    Kernel that determines if a particle is stranded under the condition that U or V == 0.
    When a particle is stranded, tt also makes sure that physical transport is set to 0 (so no updates of particles' dlon and dlat)

    Parameter Requirements
    ----------
    particle :
        - stranded - initially 0, and becomes 1 when stranded.
    fieldset :
        - `fieldset.UV`, the ocean velocities. Units [m s-1].

    Kernel Requirements
    ----------
    Order of Operations:
        At the end of physical kernels. Otherwise dlon and dlat will be updated again.

    """

    u, v = fieldset.UV[particles]

    particles.stranded = np.where((u == 0.0) | (v == 0.0), 1, particles.stranded)

    particles.dlon = np.where(particles.stranded == 1, 0.0, particles.dlon)
    particles.dlat = np.where(particles.stranded == 1, 0.0, particles.dlat)


#Kernel that determines the new weight of the particle
#Based on the maximum growth rates of morphotypes and multiple limitation curves
def sargassum_biological_growth_model(particles, fieldset):

    #We start by sampling temperature field, salinity field and nitrogen field at particle location
    particles.temperature =  fieldset.thetao[particles]
    particles.salinity =     fieldset.so[particles]

    #Selecting depth at which nitrogen field is defined
    # = particle.depth
    #if z_for_n <= 0.49402538:

    # TODO check is z_for_n is needed
    # z_for_n = 0.49402538
    # particles.nitrogen =     fieldset.no3[particles.time, z_for_n, particles.lat, particles.lon]
    particles.nitrogen =     fieldset.no3[particles]

    #RATES
    maximum_growth_rate = 0.095 #doublings/day
    mortality_rate = 0.02     #relative loss/day

    #Minimum, maximum and optimal temperature
    T_min = 20      #degC
    T_max = 31      #degC
    T_opt = 27.5    #degC

    #Nitrogen half saturation constant
    k_N  = 0.001 #mmol/m3

    #Optimal salinity
    S_opt = 36 #psu

    #GROWTH LIMITATION FUNCTION DEPENDENT ON TEMPERATURE
    #Formulation from Jouanno et al. (2025).
    limT_left = np.exp(-2 * ( (particles.temperature - T_opt)/ (T_min - T_opt))**2 )
    limT_right = np.exp(-2 * ( (particles.temperature - T_opt)/ (T_max - T_opt))**2 )
    limitation_factor_T = np.where(particles.temperature < T_opt, limT_left, limT_right)
    # if particle.temperature < T_opt:
    #     limitation_factor_T = math.exp(-2 * ( (particle.temperature - T_opt)/ (T_min - T_opt))**2 )
    # else:
    #     limitation_factor_T = math.exp(-2 * ( (particle.temperature - T_opt)/ (T_max - T_opt))**2 )

    #GROWTH LIMITATION FUNCTION DEPENDENT ON NITROGEN AVAILABILTIY
    #Formulation from Bonner et al. (2024)
    limitation_factor_N = particles.nitrogen / ( k_N + particles.nitrogen )

    #GROWTH LIMITATION FUNCTION DEPENDENT ON SALINITY
    #Formula from Jouanno et al. (2025)
    limitation_factor_S = np.exp(-0.02 * (S_opt - particles.salinity)**2 )

    ###################################

    #Save particle total limitation and seperate limitations as variables
    LIMITATION = limitation_factor_T * limitation_factor_N * limitation_factor_S
    particles.limitation = LIMITATION
    particles.lim_salinity = limitation_factor_S
    particles.lim_temp = limitation_factor_T
    particles.lim_no3 = limitation_factor_N

    #UPDATE OF PARTICLE WEIGHT with maximum specific growth rate and mortality rate converted from day-1 to s-1
    #If particle is stranded, biomass is not updated.
    #mortality_rate = 0.025 #loss/day

    # TODO check in Elena's original code what to do with stranding here
    # ptcls_stranded = particles[particles.stranded == 1]

    particles.biomass_SF3 *= 2 ** ((LIMITATION * (maximum_growth_rate / (24*60*60)) - mortality_rate / (24*60*60) ) * particles.dt )
    particles.biomass_loss = particles.biomass_SN1 - particles.biomass_SF3


def DeleteOutOfBounds(particles, fieldset):
    out_of_bounds = particles.state == parcels.StatusCode.ErrorOutOfBounds
    particles[out_of_bounds].state = parcels.StatusCode.Delete

## Executing the simulation

Now, we create an output particle file (`pfile`) and add all the `kernels` that we want to be executed during the simulation, which are formulated in the previous code. At last, we `execute` the simulation based on the satellite-based particle set (`pset`). In the execution step, the total `runtime` of the simulation is determined. It is by default set at 30 days but can be changed as prefered. The timestep of the execution (`dt`) at which the kernels are executed, should not be changed. 

**! Running this step can take up to 30 minutes**

In [ ]:
##################################################################################################
#Creating output particle file
pfile = parcels.ParticleFile(
    'Sargassum_simulation.zarr',  #file name
    outputdt=timedelta(hours=2),  #time step of the output
    chunks = (sarg_AMOUNT, 50),   #between how many timesteps data is saved
)

##################################################################################################

#Selecting kernels
kernels = [
    parcels.kernels.AdvectionRK4,
    # di_Stokes_drift,  # TODO fix Stokes drift kernel before including
    windage_drift,
    sargassum_biological_growth_model,  #biological kernel
    stranding,
    DeleteOutOfBounds,
]

##################################################################################################
#Executing the simulation
pset.execute(
    kernels,                                    #the kernels (which define how particles move)
    runtime=timedelta(days=simulation_days),    #the total runtime of the simulation
    dt=dtime_execute,                           #the timestep of the kernel executions
    output_file=pfile)                          #the output file as defined above
##################################################################################################

## Exploring and visualizing the output

In this part, we investigate the zarr data file created in the previous step and visualize the output in various ways. 

We start by opening the dataset to see what the dimensions, variables and other attributes are. 

In [ ]:
Path_to_output = "Sargassum_simulation.zarr"
Output_trajectory_dataset = xr.open_zarr(Path_to_output) #.dropna(dim='obs', how='all')  # TODO check why dropna needed

Output_trajectory_dataset

The following `biomass_plot` function is made to plot the final relative biomass of each particle and to explore the spatial variation and distribution. It can be chosen to plot it on the initial starting location (`FINAL=False`), and see how much particles will grow. Or it can be chosen to plot or the final location (`FINAL=True`) to see the distribution and relative biomass after the transport and growth. 

In [ ]:
def biomass_plot(DATA, COLORMAP, STARTTIME, FINAL=True):

    #FIGURE
    fig = plt.figure(figsize = (10,5))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.gridlines(draw_labels=['left','bottom'], zorder=2, alpha=0.3, linestyle='--')

    if FINAL == True:
        cscat = ax.scatter(DATA.lon[:,-1], DATA.lat[:,-1], c = DATA.biomass_SF3[:,-1].values,
        cmap=COLORMAP, s=1, linewidth=0, transform=ccrs.PlateCarree(), zorder=1)
    else:
    #Colored scatter plot with STARTING lon, lat and FINAL values of the categorized weight
        cscat = ax.scatter(DATA.lon[:,0], DATA.lat[:,0], c = DATA.biomass_SF3[:,-1].values,
        cmap=COLORMAP, s=2, linewidth=0, transform=ccrs.PlateCarree(), zorder=1)

    #Other figure settings
    ax.add_feature(cartopy.feature.COASTLINE.with_scale('10m'), zorder=2)
    ax.add_feature(cartopy.feature.LAND.with_scale('10m'))
    if FINAL == True:
        ax.set_title(f'Relative biomass after {simulation_days} days (released {STARTTIME.date()})')
    else:
       ax.set_title(f'Potential biomass after {simulation_days} days, plotted at initial location (released {STARTTIME.date()})')

    plt.colorbar(cscat, ax=ax, orientation='vertical', label='Sargassum relative biomass', shrink=0.65)

    return plt.show()

biomass_plot(Output_trajectory_dataset, cmo.algae, starttime, FINAL=True)

In the following piece of code, the total growth factor and seperate growth factors of temperature, nitrate availability and salinity are averaged of time for each trajectory. These time-averaged growth factors are then visualized in a boxplot to compare their relative importance in constraining growth.

In [ ]:
#Calculating mean over time of limitation values
timemean_lim_sal =    Output_trajectory_dataset['lim_salinity'].mean(dim='obs')
timemean_lim_temp =   Output_trajectory_dataset['lim_temp'].mean(dim='obs')
timemean_lim_no3 =    Output_trajectory_dataset['lim_no3'].mean(dim='obs')
timemean_lim_tot =    Output_trajectory_dataset['limitation'].mean(dim='obs')

boxplot_list = [timemean_lim_sal[:], timemean_lim_no3[:], timemean_lim_temp[:], timemean_lim_tot[:]]
print('Type of data for boxplot: ', type(boxplot_list), ' with shape: ', np.shape(boxplot_list))

limitation_names = ['Salinity', 'Nitrate', 'Temperature', 'TOTAL']
colors = ['purple' , 'indianred', 'cadetblue', 'gainsboro']

#Positions for each month
positions = np.arange(len(limitation_names))

plt.figure(figsize=(6,4), constrained_layout=True)

#Boxplots with different colors
for i, data in enumerate(boxplot_list):
    plt.boxplot(
        data,
        positions=[positions[i]],
        widths=0.5,
        patch_artist=True,
        orientation='horizontal',
        boxprops=dict(facecolor=colors[i]),
        medianprops=dict(color='black'),
        sym='+'
    )

plt.yticks(positions , limitation_names, rotation=0)
plt.xlabel('Time-averaged growth factor')
plt.title(f'Time-averaged growth factors per trajectory')
plt.xlim(-0.03,1.03)
plt.grid( linestyle='--', alpha=0.4)
plt.show()

The code in the following shows how many particles strand during the simulation. 

In [ ]:
DS = Output_trajectory_dataset

#Defining a time array in days
time_in_days = ((DS['time'] - DS['time'][:, 0]).astype("timedelta64[h]")).astype(float) / 24

#Summing the amount of stranded particles over trajectories for each time step
stranded_sum = np.sum(Output_trajectory_dataset.stranded[:, :], axis=0)

#Calculating the percentage of stranded particles relative to total amount of released particles
n_traj = len(DS.trajectory)
stranded_percentage = 100 * stranded_sum / n_traj

#Plotting over time in figure
fig, ax = plt.subplots()
ax.plot(time_in_days[0, :],stranded_percentage)
ax.set_xlabel('Time [days]')
ax.set_ylabel('Percentage of stranded particles [%]')
ax.set_title('Stranded particles after release on 01-07-2024')
ax.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

print('Final amount of stranded particles is: ', stranded_sum[-1].values, 'particles. This corresponds to ', stranded_percentage[-1].values, "%")

## References

Bonner, G., Beron-Vera, F. J., & Olascoaga, M. J. (2024). Charting the course of Sargassum: Incorporating nonlinear elastic interactions and life cycles in the Maxey–Riley model. PNAS nexus, 3(10), pgae451.

Hanisak, M. D., & Samuel, M. A. (1987, September). Growth rates in culture of several species of Sargassum from Florida, USA. In Twelfth International Seaweed Symposium: Proceedings of the Twelfth International Seaweed Symposium held in Sao Paulo, Brazil, July 27–August 1, 1986 (pp. 399-404). Dordrecht: Springer Netherlands.

Hu, C., Zhang, S., Barnes, B. B., Xie, Y., Wang, M., Cannizzaro, J. P., & English, D. C. (2023). Mapping and quantifying pelagic Sargassum in the Atlantic Ocean using multi-band medium-resolution satellite data and deep learning. Remote sensing of environment, 289, 113515.

Jouanno, J., Berthet, S., Muller-Karger, F., Aumont, O., & Sheinbaum, J. (2025). An extreme North Atlantic Oscillation event drove the pelagic Sargassum tipping point. Communications Earth & Environment, 6(1), 95.

Magaña-Gallegos, E., García-Sánchez, M., Graham, C., Olivos-Ortiz, A., Siuda, A. N., & van Tussenbroek, B. I. (2023). Growth rates of pelagic Sargassum species in the Mexican Caribbean. Aquatic Botany, 185, 103614.
